# 32 — Mẫu hình nến

61 mẫu nến TA-Lib, hai dạng kết quả, và ba chỗ dễ sai. Notebook này kết thúc
bằng phần quan trọng nhất mà hầu hết tài liệu về nến bỏ qua: **đo xem mẫu hình
có mang thông tin gì không**, thay vì kể lại ý nghĩa của nó.

1. Dạng dài và dạng rộng — khi nào dùng cái nào
2. ⚠️ **`signal` không chỉ có ±100** — và với mẫu Engulfing, `df[df.signal == 100]`
   đánh rơi **hai phần ba** số tín hiệu tăng
3. Chi phí quét cả 61 mẫu, và vì sao nêu tên mẫu cắt được 10 lần bộ nhớ
4. Event study: lợi suất sau khi mẫu hình xuất hiện, so với nền

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, hom_nay, lui_ngay, nen
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Hai dạng kết quả

**Dạng dài** (`patterns()`) — chỉ gồm các lần *bắt được*, mỗi dòng một lần.
**Dạng rộng** (`pattern.cdlXXX()`) — thêm một cột vào frame gốc, mọi phiên đều
có một giá trị.

In [2]:
gia = client.eod.stock.ohlcv(["HPG", "VCB", "FPT"], start=lui_ngay(HOM_NAY, nam=2))

dai = gia.finlens.patterns("engulfing")
rong = gia.finlens.pattern.cdlengulfing()

print(f"Frame gốc:  {len(gia):>7,} dòng")
print(f"Dạng dài:   {len(dai):>7,} dòng — chỉ các lần bắt được")
print(f"Dạng rộng:  {len(rong):>7,} dòng — bằng frame gốc, thêm cột 'cdlengulfing'")
dai.head(5)

Frame gốc:    1,494 dòng
Dạng dài:       175 dòng — chỉ các lần bắt được
Dạng rộng:    1,494 dòng — bằng frame gốc, thêm cột 'cdlengulfing'


,symbol,date,pattern,ten_mau,signal,direction
0,FPT,2024-08-14,CDLENGULFING,Engulfing Pattern,-100,giam
1,FPT,2024-09-05,CDLENGULFING,Engulfing Pattern,-80,giam
2,FPT,2024-09-26,CDLENGULFING,Engulfing Pattern,-80,giam
3,FPT,2024-09-30,CDLENGULFING,Engulfing Pattern,80,tang
4,FPT,2024-10-03,CDLENGULFING,Engulfing Pattern,-80,giam


Dạng dài có sáu cột: `symbol · date · pattern · ten_mau · signal · direction`.
Dùng nó khi bạn muốn **liệt kê tín hiệu**. Dùng dạng rộng khi bạn muốn **ghép
mẫu hình vào một pipeline tính toán khác** cùng frame.

Tên mẫu nhận cả ba cách viết:

In [3]:
for cach_viet in ["CDLDOJI", "cdldoji", "doji"]:
    print(f"  {cach_viet!r:12} → {len(gia.finlens.patterns(cach_viet)):,} dòng")

  'CDLDOJI'    → 187 dòng
  'cdldoji'    → 187 dòng
  'doji'       → 187 dòng


## 2 · ⚠️ `signal` không chỉ có ±100

Đây là cái bẫy đắt nhất của phần này, vì `df[df.signal == 100]` trông hoàn toàn
hợp lý và nó chạy không lỗi.

In [4]:
ma_hose = client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist()[:120]
lon = client.eod.stock.ohlcv(ma_hose, start=lui_ngay(HOM_NAY, nam=2.5))

print(f"Frame quét: {len(lon):,} dòng · {lon['symbol'].nunique()} mã")

tat_ca = lon.finlens.patterns()
print(f"Bắt được:   {len(tat_ca):,} dòng\n")
print(tat_ca["signal"].value_counts().sort_index().rename("số lần").to_frame().to_string())

C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


Frame quét: 72,534 dòng · 119 mã


Bắt được:   165,767 dòng

        số lần
signal        
-200       563
-100     52717
-80       7181
 80       5472
 100     99258
 200       576


In [5]:
loc_sai = tat_ca[tat_ca["signal"] == 100]
loc_dung = tat_ca[tat_ca["signal"] > 0]

print(f"df[df.signal == 100]  → {len(loc_sai):>7,} dòng")
print(f"df[df.signal > 0]     → {len(loc_dung):>7,} dòng")
print(f"\nĐánh rơi: {len(loc_dung) - len(loc_sai):,} dòng "
      f"({1 - len(loc_sai) / len(loc_dung):.1%} số tín hiệu tăng)")

print("\nNhưng con số tổng che mất chỗ đau. Chia theo từng mẫu bị ảnh hưởng:\n")
anh_huong = sorted(loc_dung[loc_dung["signal"] != 100]["pattern"].unique())
for pt in anh_huong:
    sub = loc_dung[loc_dung["pattern"] == pt]
    giu = (sub["signal"] == 100).mean()
    print(f"  {pt:<16} {len(sub):>6,} tín hiệu tăng · lọc ==100 giữ lại {giu:.0%}, "
          f"đánh rơi {1 - giu:.0%}")

df[df.signal == 100]  →  99,258 dòng
df[df.signal > 0]     → 105,306 dòng

Đánh rơi: 6,048 dòng (5.7% số tín hiệu tăng)

Nhưng con số tổng che mất chỗ đau. Chia theo từng mẫu bị ảnh hưởng:

  CDLENGULFING      2,698 tín hiệu tăng · lọc ==100 giữ lại 35%, đánh rơi 65%
  CDLHARAMI         4,364 tín hiệu tăng · lọc ==100 giữ lại 42%, đánh rơi 58%
  CDLHARAMICROSS    1,877 tín hiệu tăng · lọc ==100 giữ lại 36%, đánh rơi 64%
  CDLHIKKAKE        3,315 tín hiệu tăng · lọc ==100 giữ lại 83%, đánh rơi 17%
  CDLHIKKAKEMOD        40 tín hiệu tăng · lọc ==100 giữ lại 80%, đánh rơi 20%


Trên tổng thể chỉ khoảng 6% — nghe như không đáng lo. Nhưng con số tổng đó bị
pha loãng bởi 56 mẫu không hề phát `±80`/`±200`.

Nhìn vào từng mẫu bị ảnh hưởng thì bức tranh khác hẳn: **Engulfing mất khoảng
hai phần ba số tín hiệu tăng**, Harami cũng vậy. Nếu chiến lược của bạn dựa
trên Engulfing thì `signal == 100` xoá của bạn phần lớn tín hiệu, không phải 6%.

**`±200` đến từ đâu?** `CDLHIKKAKE` và `CDLHIKKAKEMOD` phát thêm giá trị ±200
cho **thanh xác nhận** — mẫu hình được xác nhận muộn hơn một vài phiên so với
lúc nó hình thành.

In [6]:
hik = tat_ca[tat_ca["pattern"].str.contains("HIKKAKE")]
print(f"Các mẫu phát ±200: {sorted(hik['pattern'].unique())}")
print(f"\nPhân bố signal của riêng nhóm này:")
print(hik["signal"].value_counts().sort_index().to_string())

Các mẫu phát ±200: ['CDLHIKKAKE', 'CDLHIKKAKEMOD']

Phân bố signal của riêng nhóm này:
signal
-200     563
-100    2393
 100    2779
 200     576


**`±80` đến từ đâu?** Một số mẫu TA-Lib phát tín hiệu ở hai mức độ tin cậy.

In [7]:
tam_muoi = tat_ca[tat_ca["signal"].abs() == 80]
print(f"Các mẫu phát ±80: {sorted(tam_muoi['pattern'].unique())}")

Các mẫu phát ±80: ['CDLENGULFING', 'CDLHARAMI', 'CDLHARAMICROSS']


### Cách lọc đúng

Dùng **dấu**, hoặc dùng cột `direction` đã suy sẵn:

In [8]:
print("Ba cách tương đương nhau:")
a = len(tat_ca[tat_ca["signal"] > 0])
b = len(tat_ca[tat_ca["direction"] == "tang"])
c = len(tat_ca[tat_ca["signal"].isin([80, 100, 200])])
print(f"  signal > 0          : {a:,}")
print(f"  direction == 'tang' : {b:,}")
print(f"  signal in (80,100,200): {c:,}")
print(f"\nBa cách cho cùng kết quả: {a == b == c}")

Ba cách tương đương nhau:
  signal > 0          : 105,306
  direction == 'tang' : 105,306
  signal in (80,100,200): 105,306

Ba cách cho cùng kết quả: True


Cột `direction` tồn tại chính vì lý do này — nó là câu trả lời không cần bạn
nhớ tập giá trị nào hợp lệ.

## 3 · Chi phí quét cả 61 mẫu

Quét tất cả không phải một lựa chọn trung tính. Mỗi dòng đầu vào sinh ra hơn
hai dòng đầu ra.

In [9]:
import time

t0 = time.perf_counter()
tat = lon.finlens.patterns()
t_tat = time.perf_counter() - t0
bo_nho_tat = tat.memory_usage(deep=True).sum() / 1024**2

t0 = time.perf_counter()
mot_vai = lon.finlens.patterns(["engulfing", "morningstar", "hammer"])
t_vai = time.perf_counter() - t0
bo_nho_vai = mot_vai.memory_usage(deep=True).sum() / 1024**2

print(f"{'':22} {'dòng':>10} {'giây':>7} {'MiB':>8}")
print(f"{'quét cả 61 mẫu':22} {len(tat):>10,} {t_tat:>7.2f} {bo_nho_tat:>8.1f}")
print(f"{'nêu tên 3 mẫu':22} {len(mot_vai):>10,} {t_vai:>7.2f} {bo_nho_vai:>8.1f}")
print(f"\nTỷ lệ: {len(tat) / len(mot_vai):.0f}× số dòng, {bo_nho_tat / bo_nho_vai:.0f}× bộ nhớ")
print(f"Mỗi dòng đầu vào sinh {len(tat) / len(lon):.2f} dòng kết quả")

                             dòng    giây      MiB
quét cả 61 mẫu            165,767    0.38     38.1
nêu tên 3 mẫu              10,609    0.04      2.4

Tỷ lệ: 16× số dòng, 16× bộ nhớ
Mỗi dòng đầu vào sinh 2.29 dòng kết quả


**Nêu tên mẫu trong `which=` cắt được hàng chục lần bộ nhớ, không phải vài phần
trăm.** Ở quy mô toàn sàn nhiều năm, đó là khác biệt giữa chạy được và hết RAM.

## 4 · Mẫu nào bắt được nhiều nhất — và vì sao điều đó quan trọng

In [10]:
tan_suat = (
    tat.groupby("ten_mau", observed=True)
    .size()
    .div(len(lon))
    .mul(100)
    .sort_values(ascending=False)
    .head(15)
    .round(2)
    .reset_index()
)
tan_suat.columns = ["mẫu hình", "tần suất %"]

bar_ngang(
    tan_suat,
    nhan="mẫu hình",
    gia_tri="tần suất %",
    tieu_de="15 mẫu nến xuất hiện nhiều nhất",
    phu_de="% số phiên bắt được mẫu, tính trên toàn bộ frame quét",
    nhan_x="% số phiên",
    dinh_dang_nhan="{:.2f}%",
)

Doji xuất hiện ở **khoảng một phần tư** số phiên. Một tín hiệu xuất hiện thường
xuyên đến thế thì không thể mang nhiều thông tin — nếu nó dự báo được điều gì,
điều đó đã xảy ra một phần tư số phiên rồi.

Đây là **tần suất nền** (base rate), và nó là thứ phải trừ đi trước khi kết
luận bất cứ điều gì về một mẫu hình.

## 5 · Event study — mẫu hình có mang thông tin không?

Câu hỏi đúng không phải "mẫu này nghĩa là gì" mà là: **sau khi mẫu này xuất
hiện, lợi suất trung bình có khác với lợi suất nền không?**

Phương pháp:

1. Tính lợi suất tương lai 1, 3, 5, 10 phiên cho **mọi** phiên
2. Lấy trung bình lợi suất tương lai ở các phiên **có** mẫu hình
3. So với trung bình của **toàn bộ** phiên (đường nền)
4. Chia chênh lệch cho sai số chuẩn để biết nó có vượt qua nhiễu không

⚠️ Lợi suất tương lai phải tính bằng `shift(-k)` **trong từng mã**. Quên
`groupby` là để giá của mã sau rò vào mã trước — cùng một lỗi rò rỉ ranh giới
nhóm như ở notebook `31`.

In [11]:
KHUNG = [1, 3, 5, 10]

nen_gia = lon.sort_values(["symbol", "date"]).copy()
for k in KHUNG:
    nen_gia[f"ls_{k}"] = (
        nen_gia.groupby("symbol", observed=True)["close"].shift(-k) / nen_gia["close"] - 1
    ) * 100

print("Vì sao KHÔNG dùng trung vị làm đường nền ở đây:")
for k in KHUNG:
    r = nen_gia[f"ls_{k}"]
    print(f"  {k:>2} phiên: trung vị {r.median():+.4f}%  ·  "
          f"trung bình {r.mean():+.4f}%  ·  tỷ lệ lợi suất đúng bằng 0: {(r == 0).mean():.1%}")

Vì sao KHÔNG dùng trung vị làm đường nền ở đây:
   1 phiên: trung vị +0.0000%  ·  trung bình +0.0146%  ·  tỷ lệ lợi suất đúng bằng 0: 19.1%
   3 phiên: trung vị +0.0000%  ·  trung bình +0.0400%  ·  tỷ lệ lợi suất đúng bằng 0: 9.2%
   5 phiên: trung vị +0.0000%  ·  trung bình +0.0567%  ·  tỷ lệ lợi suất đúng bằng 0: 6.8%
  10 phiên: trung vị -0.2301%  ·  trung bình +0.0844%  ·  tỷ lệ lợi suất đúng bằng 0: 4.5%


Trung vị lợi suất 1 phiên bằng **đúng 0** vì gần 20% số phiên cổ phiếu đứng giá
— biên độ dao động nhỏ cộng bước giá tối thiểu làm phân phối có một khối lượng
lớn dồn ở đúng 0. Một đường nền bằng 0 không phân biệt được gì cả.

Nên phần dưới dùng **trung bình**, và kèm sai số chuẩn để biết chênh lệch có
vượt qua nhiễu hay không.

In [12]:
duong_nen = {k: nen_gia[f"ls_{k}"].mean() for k in KHUNG}
print("Đường nền — trung bình lợi suất tương lai của MỌI phiên:")
for k, v in duong_nen.items():
    print(f"  {k:>2} phiên: {v:+.4f}%")

Đường nền — trung bình lợi suất tương lai của MỌI phiên:
   1 phiên: +0.0146%
   3 phiên: +0.0400%
   5 phiên: +0.0567%
  10 phiên: +0.0844%


In [13]:
MAU_XET = ["engulfing", "hammer", "morningstar", "eveningstar", "3whitesoldiers", "shootingstar"]

ket = []
for ten_mau in MAU_XET:
    bat = lon.finlens.patterns(ten_mau)
    if bat.empty:
        continue
    for huong in ["tang", "giam"]:
        loc = bat[bat["direction"] == huong][["symbol", "date"]]
        if len(loc) < 50:  # dưới 50 lần thì trung vị không đáng tin
            continue
        ghep = loc.merge(nen_gia, on=["symbol", "date"], how="inner")
        dong = {"mẫu": ten_mau, "hướng": huong, "số lần": len(ghep)}
        for k in KHUNG:
            r = ghep[f"ls_{k}"].dropna()
            vuot = r.mean() - duong_nen[k]
            sai_so = r.std() / np.sqrt(len(r))  # sai số chuẩn của trung bình
            dong[f"{k}p"] = round(vuot, 3)
            dong[f"{k}p / sai số"] = round(vuot / sai_so, 2) if sai_so > 0 else np.nan
        ket.append(dong)

vuot_nen = pd.DataFrame(ket).sort_values("5p", ascending=False)
print("Lợi suất trung bình VƯỢT đường nền (điểm phần trăm),")
print("kèm tỷ số vượt-nền trên sai số chuẩn — |tỷ số| < 2 nghĩa là không phân biệt được với nhiễu:\n")
vuot_nen

Lợi suất trung bình VƯỢT đường nền (điểm phần trăm),
kèm tỷ số vượt-nền trên sai số chuẩn — |tỷ số| < 2 nghĩa là không phân biệt được với nhiễu:



,mẫu,hướng,số lần,1p,1p / sai số,3p,3p / sai số,5p,5p / sai số,10p,10p / sai số
1,engulfing,giam,5592,0.092,3.21,0.198,4.12,0.130,2.13,0.169,1.93
5,shootingstar,giam,599,-0.058,-0.68,-0.054,-0.32,-0.030,-0.14,-0.060,-0.19
0,engulfing,tang,2698,0.001,0.02,-0.093,-1.29,-0.120,-1.29,-0.199,-1.55
2,hammer,tang,2193,-0.137,-3.43,-0.157,-2.27,-0.352,-3.98,-0.332,-2.64
4,eveningstar,giam,177,-0.376,-2.01,-0.338,-1.03,-0.411,-0.97,-0.103,-0.19
3,morningstar,tang,126,-0.099,-0.43,-0.717,-1.95,-0.437,-0.78,-0.571,-0.85


Đọc bảng này cho đúng. Cột `5p` là **chênh lệch so với nền**, không phải lợi
suất: `+0.05` nghĩa là sau mẫu hình đó, lợi suất trung bình 5 phiên cao hơn một
phiên bất kỳ đúng 0,05 điểm phần trăm.

Cột `5p / sai số` mới là cột quyết định. Nó là chênh lệch chia cho sai số chuẩn
— cùng ý tưởng với thống kê t. **Trị tuyệt đối dưới 2 thì chênh lệch đó không
phân biệt được với nhiễu lấy mẫu**, dù dấu của nó có "đúng lý thuyết" đến đâu.

In [14]:
ve = vuot_nen.assign(nhan=lambda d: d["mẫu"] + " · " + d["hướng"])

fig = go.Figure()
for i, k in enumerate(KHUNG):
    fig.add_trace(
        go.Bar(
            x=ve["nhan"],
            y=ve[f"{k}p"],
            name=f"{k} phiên",
            marker=dict(color=CHUOI[i], line=dict(color="#fcfcfb", width=2)),
            hovertemplate="%{x}<br>%{y:+.3f} đpt<extra>" + f"{k} phiên</extra>",
        )
    )
fig.add_hline(y=0, line_width=1, line_color="#c3c2b7")
fig.update_layout(
    barmode="group",
    title_text="Lợi suất vượt nền sau mẫu hình nến<br>"
    "<sub style='color:#52514e'>Trung vị lợi suất sau mẫu, trừ đi trung vị của mọi phiên · đơn vị điểm phần trăm</sub>",
    yaxis_title="điểm phần trăm so với nền",
    height=520,
    xaxis=dict(tickangle=-25),
)
fig

### Kết luận trung thực

Phần lớn các dòng có |tỷ số| dưới 2 — tức chênh lệch không phân biệt được với
nhiễu. Nhưng điều thú vị nằm ở hai dòng **có** vượt ngưỡng: chúng vượt theo
chiều **ngược với tên gọi của mẫu**.

Engulfing *giảm* — một mẫu đảo chiều xuống theo sách vở — lại đi kèm lợi suất
1 phiên **cao hơn** nền. Hammer *tăng* — mẫu đảo chiều lên — đi kèm lợi suất
**thấp hơn** nền. Cả hai đều đạt |tỷ số| > 3.

Cách đọc hợp lý nhất: những mẫu này đánh dấu **phiên có biến động mạnh**, và
sau một phiên biến động mạnh thì giá có xu hướng hồi lại một phần — hiện tượng
đảo chiều ngắn hạn quen thuộc. Mẫu hình đang bắt được cái đó, chứ không phải
bắt được "tâm lý đảo chiều" như cách nó thường được kể.

Ba điều mang về:

- **Mẫu nến đơn lẻ không mang biên lợi thế theo chiều mà tên nó gợi ra.** Chỗ
  có tín hiệu thì tín hiệu ngược dấu.
- Cách dùng có nghĩa hơn là **làm bộ lọc trong một hệ thống**: mẫu đảo chiều
  *tại một ngưỡng hỗ trợ*, *khi khối lượng đột biến*, *sau một chuỗi giảm* —
  mẫu hình cộng điều kiện, không phải mẫu hình một mình.
- Bất cứ ai đưa bạn một mẫu hình kèm "tỷ lệ thắng 78%" mà không nói tỷ lệ nền
  là bao nhiêu thì đang bán cho bạn một con số vô nghĩa.

⚠️ Ba giới hạn của phép đo này, phải nói rõ: nó dùng **một** khung mẫu (120 mã,
2,5 năm), **một** giai đoạn thị trường, và **không** tính chi phí giao dịch.
Nó đủ để bác bỏ tuyên bố "mẫu này rất mạnh theo chiều X"; nó không đủ để khẳng
định chiều ngược lại là một chiến lược.

## 6 · Nhìn tận mắt một mẫu hình

Cuối cùng, một việc rất đơn giản mà ai làm phân tích kỹ thuật cũng nên làm:
tìm một lần bắt được và **nhìn vào biểu đồ nến ở chỗ đó**.

In [15]:
MA = "HPG"
mot_ma = client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, nam=1)).sort_values("date")

bat_duoc = mot_ma.finlens.patterns("engulfing")
gan_nhat = bat_duoc.iloc[-1]
print(f"Lần gần nhất: {gan_nhat['ten_mau']} · {gan_nhat['date']:%d/%m/%Y} · "
      f"signal={gan_nhat['signal']} ({gan_nhat['direction']})")
if abs(gan_nhat["signal"]) != 100:
    print(f"\n⚠️ signal={gan_nhat['signal']}, không phải ±100 — "
          "một bộ lọc `== 100` sẽ bỏ qua đúng lần bắt gần nhất này.")

cua_so = mot_ma[
    (mot_ma["date"] >= gan_nhat["date"] - pd.Timedelta(days=30))
    & (mot_ma["date"] <= gan_nhat["date"] + pd.Timedelta(days=20))
]

fig = nen(
    cua_so,
    tieu_de=f"{MA} — {gan_nhat['ten_mau']} ngày {gan_nhat['date']:%d/%m/%Y}",
    phu_de=f"signal={gan_nhat['signal']} · hướng {gan_nhat['direction']}",
)
fig.add_vline(
    x=gan_nhat["date"],
    line_width=2,
    line_dash="dot",
    line_color=TANG if gan_nhat["direction"] == "tang" else GIAM,
    row=1,
    col=1,
)
fig

Lần gần nhất: Engulfing Pattern · 03/08/2026 · signal=80 (tang)

⚠️ signal=80, không phải ±100 — một bộ lọc `== 100` sẽ bỏ qua đúng lần bắt gần nhất này.


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Liệt kê tín hiệu | `df.finlens.patterns("engulfing")` — dạng dài |
| Thêm cột vào pipeline | `df.finlens.pattern.cdlengulfing()` — dạng rộng |
| Nhiều mẫu cùng lúc | `df.finlens.patterns(["engulfing", "hammer"])` |
| Chỉ tín hiệu tăng | `df[df.direction == "tang"]` — **không** `df.signal == 100` |

**Bốn điều mang sang notebook sau:**

1. ⚠️ `df[df.signal == 100]` đánh rơi ~6% tín hiệu tăng trên tổng thể — nhưng
   **hai phần ba** nếu mẫu bạn dùng là Engulfing hay Harami. Lọc bằng
   `direction` hoặc bằng dấu, không bao giờ bằng `== 100`.
2. Quét cả 61 mẫu tốn hàng chục lần bộ nhớ so với nêu tên. `which=` không phải
   một tối ưu vặt.
3. Doji xuất hiện ở ~25% số phiên. **Luôn trừ tần suất nền** trước khi kết luận.
4. Mẫu nến đơn lẻ không mang biên lợi thế đủ để giao dịch — chênh lệch so với
   nền không vượt qua sai số lấy mẫu. Dùng nó làm **bộ lọc trong một hệ
   thống**, không dùng làm tín hiệu độc lập.

---

**Tiếp theo:** [`33_screener_tin_hieu.ipynb`](33_screener_tin_hieu.ipynb) — quét
tín hiệu kỹ thuật trên toàn sàn HOSE.